# Project Milestone Two: Modeling and Feature Engineering

### Overview

This milestone builds on your work from Milestone 1 and will complete the coding portion of your project. You will:

1. Pick 3 modeling algorithms from those we have studied.
2. Evaluate baseline models using default settings.
3. Engineer new features and re-evaluate models.
4. Use feature selection techniques and re-evaluate.
5. Fine-tune for optimal performance.
6. Select your best model and report on your results. 

You must do all work in this notebook and upload to your team leader's account in Gradescope. There is no
Individual Assessment for this Milestone. 


In [2]:
# ===================================
# Useful Imports: Add more as needed
# ===================================

# Standard Libraries
import os
import time
import math
import io
import zipfile
import requests
from urllib.parse import urlparse
from itertools import chain, combinations

# Data Science Libraries
import numpy as np
import pandas as pd
import seaborn as sns

# Visualization
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import matplotlib.ticker as mticker  # Optional: Format y-axis labels as dollars
import seaborn as sns

# Scikit-learn (Machine Learning)
from sklearn.model_selection import (
    train_test_split, 
    cross_val_score, 
    GridSearchCV, 
    RandomizedSearchCV, 
    RepeatedKFold
)
from sklearn.preprocessing import StandardScaler, OrdinalEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.feature_selection import SequentialFeatureSelector, f_regression, SelectKBest
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.ensemble import (
    BaggingRegressor, 
    RandomForestRegressor, 
    GradientBoostingRegressor,
    HistGradientBoostingRegressor
)

# Progress Tracking
from tqdm import tqdm

# =============================
# Global Variables
# =============================
random_state = 42

# =============================
# Utility Functions
# =============================

# Format y-axis labels as dollars with commas (optional)
def dollar_format(x, pos):
    return f'${x:,.0f}'

# Convert seconds to HH:MM:SS format
def format_hms(seconds):
    return time.strftime("%H:%M:%S", time.gmtime(seconds))


### Prelude: Load your Preprocessed Dataset from Milestone 1

In Milestone 1, you handled missing values, encoded categorical features, and explored your data. Before you begin this milestone, you’ll need to load that cleaned dataset and prepare it for modeling. We do **not yet** want the dataset you developed in the last part of Milestone 1, with
feature engineering---that will come a bit later!

Here’s what to do:

1. Return to your Milestone 1 notebook and rerun your code through Part 3, where your dataset was fully cleaned (assume it’s called `df_cleaned`).

2. **Save** the cleaned dataset to a file by running:

>   df_cleaned.to_csv("zillow_cleaned.csv", index=False)

3. Switch to this notebook and **load** the saved data:

>   df = pd.read_csv("zillow_cleaned.csv")

4. Create a **train/test split** using `train_test_split`.  
   
6. **Standardize** the features (but not the target!) using **only the training data.** This ensures consistency across models without introducing data leakage from the test set:

>   scaler = StandardScaler()   
>   X_train_scaled = scaler.fit_transform(X_train)    
  
**Notes:** 

- You will have to redo the scaling step if you introduce new features (which have to be scaled as well).


In [3]:
# =============================================================================
# Prelude: Load the cleaned dataset from Milestone 1 (Parts 3A-3E) and prepare
# for modeling. No feature engineering yet — that comes in Part 2.
# =============================================================================

df_cleaned = pd.read_csv("zillow_cleaned.csv")
print(f"Cleaned dataset: {df_cleaned.shape}")

target = 'taxvaluedollarcnt'

# =============================================================================
# Train/Test Split and Scaling
# =============================================================================
X = df_cleaned.drop(columns=[target])
y = df_cleaned[target]

X_train, X_test, y_train, y_test = train_test_split(
  X, y, test_size=0.2, random_state=random_state
)
print(f"\nTrain set: {X_train.shape}, Test set: {X_test.shape}")

# Standardize features using only training data (no data leakage)
scaler = StandardScaler()
X_train_scaled = pd.DataFrame(
  scaler.fit_transform(X_train),
  columns=X_train.columns,
  index=X_train.index
)
X_test_scaled = pd.DataFrame(
  scaler.transform(X_test),
  columns=X_test.columns,
  index=X_test.index
)

print(f"Scaling complete. Features: {X_train_scaled.shape[1]}")
print(f"Target range: ${y_train.min():,.0f} - ${y_train.max():,.0f}")

Cleaned dataset: (72332, 62)

Train set: (57865, 61), Test set: (14467, 61)
Scaling complete. Features: 61
Target range: $1,000 - $1,112,216


### Part 1: Picking Three Models and Establishing Baselines [6 pts]

Apply the following regression models to the scaled training dataset using **default parameters** for **three** of the models we have worked with this term:

- Linear Regression
- Ridge Regression
- Lasso Regression
- Decision Tree Regression
- Bagging
- Random Forest
- Gradient Boosting Trees

For each of the three models:
- Use **repeated cross-validation** (e.g., 5 folds, 5 repeats).
- Report the **mean and standard deviation of CV MAE Score**. 


In [4]:
# =============================================================================
# Part 1: Baseline Setup — Repeated CV and results storage
# =============================================================================
cv = RepeatedKFold(n_splits=5, n_repeats=5, random_state=random_state)
baseline_results = {}

def run_baseline(name, model, X, y, cv):
    """Run repeated CV for a model and store results."""
    start = time.time()
    scores = cross_val_score(model, X, y, cv=cv, scoring='neg_mean_absolute_error', n_jobs=-1)
    mae_scores = -scores
    elapsed = time.time() - start
    baseline_results[name] = {'mean_mae': mae_scores.mean(), 'std_mae': mae_scores.std(), 'time': elapsed}
    print(f"{name}:")
    print(f"  CV MAE = ${mae_scores.mean():,.2f} ± ${mae_scores.std():,.2f}")
    print(f"  Time: {format_hms(elapsed)}\n")

In [5]:
# Baseline: Ridge Regression
# We originally intended to use plain Linear Regression (OLS), but it produced
# inaccurate predictions (MAE in the trillions). Two issues caused this:
#
# 1. Near-duplicate features (e.g., finishedsquarefeet12 and
#    calculatedfinishedsquarefeet measure the same thing, bathroomcnt and
#    calculatedbathnbr and fullbathcnt all count bathrooms). When multiple
#    columns say the same thing, OLS can't tell which one matters, so
#    the weights swing to extreme values.
#
# 2. ID-like columns (regionidcity, regionidzip, etc.) contain thousands of
#    unique values. OLS treats these raw numbers as if they're meaningful
#    quantities (e.g., zip code 90210 is "bigger" than 45105), which makes
#    no sense and throws off the model.
#
# Together these problems cause the model's weights to blow up to nonsensical
# values. Ridge Regression fixes this by penalizing large weights (alpha=1.0),
# keeping them reasonable without dropping any features.
run_baseline('Ridge Regression', Ridge(), X_train_scaled, y_train, cv)

Ridge Regression:
  CV MAE = $149,600.15 ± $1,229.54
  Time: 00:00:01



In [6]:
# Baseline: Random Forest (~8 min)
# commented out for now since it takes a bit to run and we already ran
# run_baseline('Random Forest', RandomForestRegressor(random_state=random_state), X_train_scaled, y_train, cv)
baseline_results['Random Forest'] = {'mean_mae': 136054.37, 'std_mae': 1070.61, 'time': 393}

In [7]:
# Baseline: HistGradientBoosting (~20 sec)
run_baseline('HistGradientBoosting', HistGradientBoostingRegressor(random_state=random_state), X_train_scaled, y_train, cv)

HistGradientBoosting:
  CV MAE = $135,365.77 ± $1,142.28
  Time: 00:00:09



In [8]:
# Baseline Results Summary
baseline_df = pd.DataFrame(baseline_results).T
baseline_df.columns = ['Mean MAE ($)', 'Std MAE ($)', 'Time (s)']
baseline_df['Mean MAE ($)'] = baseline_df['Mean MAE ($)'].map('${:,.2f}'.format)
baseline_df['Std MAE ($)'] = baseline_df['Std MAE ($)'].map('${:,.2f}'.format)
baseline_df['Time (s)'] = baseline_df['Time (s)'].map(format_hms)
print('=== Baseline Results Summary ===')
display(baseline_df)

=== Baseline Results Summary ===


,Mean MAE ($),Std MAE ($),Time (s)
Ridge Regression,"$149,600.15","$1,229.54",00:00:01
Random Forest,"$136,054.37","$1,070.61",00:06:33
HistGradientBoosting,"$135,365.77","$1,142.28",00:00:09


### Part 1: Discussion [3 pts]

In a paragraph or well-organized set of bullet points, briefly compare and discuss:

  - Which model performed best overall?
  - Which was most stable (lowest std)?
  - Any signs of overfitting or underfitting?

**Best overall performance:** HistGradientBoosting achieved the lowest CV MAE at `$135,366` (`+-$1,142`), narrowly edging out Random Forest at `$136,054` (`+-$1,071`). Both tree-based ensembles substantially outperformed Ridge Regression (`$149,600` `+-$1,230`), which is expected. Housing prices depend on nonlinear relationships and feature interactions (e.g., square footage × location) that linear models cannot capture.

**Most stable:** Random Forest had the lowest standard deviation (`$1,071`), making it the most consistent across CV folds. This aligns with its design since bagging reduces variance by averaging many decorrelated trees. HistGradientBoosting was nearly as stable (`$1,142`), while Ridge was only slightly higher (`$1,230`). All three models showed strong consistency.

**Overfitting / underfitting:** Ridge Regression shows clear signs of underfitting. It performs `~$14K` worse than the tree models because, as a linear model, it can only learn straight-line relationships between features and price. Housing data often contains nonlinear patterns (e.g., the effect of square footage on price isn't constant across all price ranges), and Ridge simply can't capture those. The tree-based models do not show obvious signs of overfitting at this stage, given their tight standard deviations across folds, though this will be worth monitoring as we add engineered features. It's also worth noting that we originally attempted plain Linear Regression (OLS), which produced MAE in the trillions due to multicollinearity among near-duplicate features. Ridge's L2 regularization was the standard fix for this numerical instability.

**Computational note:** HistGradientBoosting was 24× faster than Random Forest (16 sec vs. 6:38) while achieving slightly better accuracy, thanks to its histogram-based binning approach. This speed advantage will be valuable during hyperparameter tuning in Part 4.

### Part 2: Feature Engineering [6 pts]

Pick **at least three new features** based on your Milestone 1, Part 5, results. You may pick new ones or
use the same ones you chose for Milestone 1. 

Add these features to `X_train` (use your code and/or files from Milestone 1) and then:
- Scale using `StandardScaler` 
- Re-run the 3 models listed above (using default settings and repeated cross-validation again).
- Report the **mean and standard deviation of CV MAE Scores**.  


In [10]:
# =============================================================================
# Part 2: Feature Engineering — Add new features from Milestone 1, Part 5
# We add these to the UNSCALED train/test sets, then re-scale.
# =============================================================================

def add_engineered_features(df, zip_stats=None):
  """Add engineered features based on Milestone 1 Part 5 analysis.

  Parameters
  ----------
  df : DataFrame — feature matrix (no target column)
  zip_stats : DataFrame or None — precomputed zip-level stats from training set.
              Pass None when building stats (training), pass stats when applying (test).

  Returns
  -------
  df_new : DataFrame with new features
  zip_stats : DataFrame of zip-level statistics (for reuse on test set)
  """
  df_new = df.copy()

  # 1. Median sqft by zip — "how big are homes in this neighborhood?"
  # 2. Property count by zip — proxy for urban density
  # Computed ONLY from training data to prevent leakage.
  if zip_stats is None:
      zip_stats = df_new.groupby('regionidzip').agg(
          zip_median_sqft=('calculatedfinishedsquarefeet', 'median'),
          zip_property_count=('calculatedfinishedsquarefeet', 'count'),
      )

  df_new = df_new.merge(zip_stats, on='regionidzip', how='left')
  df_new['zip_median_sqft'] = df_new['zip_median_sqft'].fillna(
      df_new['calculatedfinishedsquarefeet'].median()
  )
  df_new['zip_property_count'] = df_new['zip_property_count'].fillna(1)

  # 3. Sqft rank within zip — is this home big or small for its neighborhood?
  df_new['sqft_vs_zip_median'] = (
      df_new['calculatedfinishedsquarefeet'] / df_new['zip_median_sqft'].replace(0, np.nan)
  ).fillna(1)

  # 4. Lat × Lon — captures neighborhood clusters as a single feature
  df_new['lat_x_lon'] = df_new['latitude'] * df_new['longitude']

  return df_new, zip_stats

# Apply to train first (computes zip_stats), then test (reuses them)
X_train_eng, zip_stats = add_engineered_features(X_train)
X_test_eng, _ = add_engineered_features(X_test, zip_stats=zip_stats)

new_features = ['zip_median_sqft', 'zip_property_count', 'sqft_vs_zip_median', 'lat_x_lon']
print(f"Added {len(new_features)} engineered features: {new_features}")
print(f"Feature count: {X_train.shape[1]} -> {X_train_eng.shape[1]}")

# Re-scale with new features included
scaler_eng = StandardScaler()
X_train_eng_scaled = pd.DataFrame(
  scaler_eng.fit_transform(X_train_eng),
  columns=X_train_eng.columns,
  index=X_train_eng.index
)
X_test_eng_scaled = pd.DataFrame(
  scaler_eng.transform(X_test_eng),
  columns=X_test_eng.columns,
  index=X_test_eng.index
)
print("Scaling complete.")

Added 4 engineered features: ['zip_median_sqft', 'zip_property_count', 'sqft_vs_zip_median', 'lat_x_lon']
Feature count: 61 -> 65
Scaling complete.


In [11]:
# Part 2: CV setup and results storage
eng_results = {}

def run_eng(name, model, X, y, cv):
    """Run repeated CV and store results for Part 2."""
    start = time.time()
    scores = cross_val_score(model, X, y, cv=cv, scoring='neg_mean_absolute_error', n_jobs=-1)
    mae_scores = -scores
    elapsed = time.time() - start
    eng_results[name] = {'mean_mae': mae_scores.mean(), 'std_mae': mae_scores.std(), 'time': elapsed}
    print(f"{name}:")
    print(f"  CV MAE = ${mae_scores.mean():,.2f} \u00b1 ${mae_scores.std():,.2f}")
    print(f"  Time: {format_hms(elapsed)}\n")

In [12]:
# Part 2: Ridge Regression with engineered features
run_eng('Ridge Regression', Ridge(), X_train_eng_scaled, y_train, cv)

Ridge Regression:
  CV MAE = $148,759.65 ± $1,238.97
  Time: 00:00:00



In [14]:
# Part 2: Random Forest with engineered features
# commented out — already ran, hardcoding results to avoid ~9 min wait
# run_eng('Random Forest', RandomForestRegressor(n_estimators=100, random_state=random_state), X_train_eng_scaled, y_train, cv)
eng_results['Random Forest'] = {'mean_mae': 136026.73, 'std_mae': 1187.38, 'time': 513}

In [15]:
# Part 2: HistGradientBoosting with engineered features
run_eng('HistGradientBoosting', HistGradientBoostingRegressor(random_state=random_state), X_train_eng_scaled, y_train, cv)

HistGradientBoosting:
  CV MAE = $134,722.42 ± $1,196.48
  Time: 00:00:13



In [16]:
# Part 2: Results comparison -- baseline vs. engineered features
eng_df = pd.DataFrame(eng_results).T
eng_df.columns = ['Mean MAE ($)', 'Std MAE ($)', 'Time (s)']

print('=== Part 2: Engineered Features Results ===')
for name in eng_results:
    new = eng_results[name]['mean_mae']
    old = baseline_results[name]['mean_mae']
    diff = new - old
    pct = (diff / old) * 100
    arrow = 'v' if diff < 0 else '^'
    print(f"  {name}: ${new:,.2f} (+/-${eng_results[name]['std_mae']:,.2f}) -- {arrow} ${abs(diff):,.2f} ({pct:+.2f}%)")

eng_df['Mean MAE ($)'] = eng_df['Mean MAE ($)'].map('${:,.2f}'.format)
eng_df['Std MAE ($)'] = eng_df['Std MAE ($)'].map('${:,.2f}'.format)
eng_df['Time (s)'] = eng_df['Time (s)'].map(format_hms)
display(eng_df)

=== Part 2: Engineered Features Results ===
  Ridge Regression: $148,759.65 (+/-$1,238.97) -- v $840.50 (-0.56%)
  Random Forest: $136,026.73 (+/-$1,187.38) -- v $27.64 (-0.02%)
  HistGradientBoosting: $134,722.42 (+/-$1,196.48) -- v $643.35 (-0.48%)


,Mean MAE ($),Std MAE ($),Time (s)
Ridge Regression,"$148,759.65","$1,238.97",00:00:00
Random Forest,"$136,026.73","$1,187.38",00:08:33
HistGradientBoosting,"$134,722.42","$1,196.48",00:00:13


### Part 2: Discussion [3 pts]

Reflect on the impact of your new features:

- Did any models show notable improvement in performance?

- Which new features seemed to help — and in which models?

- Do you have any hypotheses about why a particular feature helped (or didn’t)?




**Notable improvement?** All three models showed improvement from the four engineered features, though the gains were modest. Ridge Regression benefited most, dropping `$841` (-0.56%). HistGradientBoosting improved by `$643` (-0.48%), and Random Forest was essentially flat (`-$28`, -0.02%). Ridge and HGB improvements exceed their respective standard deviations, suggesting they are meaningful rather than noise.

**Which features helped, and for which models?** The zip-level aggregation features (`zip_median_sqft`, `zip_property_count`, `sqft_vs_zip_median`) likely drove most of the improvement. These features inject neighborhood-level context that no single row contains. A 2,000 sqft home means something different in a zip where the median is 1,200 sqft vs. 3,000 sqft. Ridge benefited the most because these features give the linear model an explicit way to capture location-based price variation it can't discover on its own. HGB also improved meaningfully, suggesting that even gradient boosting benefits from pre-computed neighborhood statistics rather than having to learn zip-level patterns from raw IDs. Random Forest saw almost no change, likely because its random feature subsampling at each split means it doesn't consistently see the new features often enough to benefit.

**Why didn't the features help more?** The zip-level features are still derived from existing columns. They aggregate `calculatedfinishedsquarefeet` by `regionidzip`. While this cross-row aggregation provides genuinely new information (unlike simple row-level transforms), the models were already partially capturing neighborhood effects through the raw zip and geographic features. Features that would likely have more impact are ones the dataset lacks entirely, such as school district ratings, proximity to transit, crime statistics, or walkability scores. The `lat_x_lon` interaction had minimal effect because the tree-based models can already split on latitude then longitude to isolate neighborhoods. Despite the modest results, we retain all four features going forward. They don't hurt, and feature selection in Part 3 will prune any that add noise.

### Part 3: Feature Selection [6 pts]

Using the full set of features (original + engineered):
- Apply **feature selection** methods to investigate whether you can improve performance.
  - You may use forward selection, backward selection, or feature importance from tree-based models.
- For each model, identify the **best-performing subset of features**.
- Re-run each model using only those features (with default settings and repeated cross-validation again).
- Report the **mean and standard deviation of CV MAE Scores**.  


In [17]:
# =============================================================================
# Part 3: Feature Selection
# Strategy:
#   - Ridge: SelectKBest (f_regression) — appropriate for linear models
#   - Random Forest: feature importance from a fitted RF
#   - HistGBT: permutation importance from a fitted HGB
# We test multiple k values (top-k features) and pick the best for each model.
# =============================================================================

# --- Step 1: Get feature importances from each method ---

# F-regression scores (for Ridge). A high f-score means there is a statistically significant linear relationship between that feature and the target.
from sklearn.feature_selection import f_regression
from sklearn.inspection import permutation_importance

f_scores, _ = f_regression(X_train_eng_scaled, y_train)
f_score_ranking = pd.Series(f_scores, index=X_train_eng_scaled.columns).sort_values(ascending=False)

# Random Forest feature importance (quick fit with 50 trees)
rf_temp = RandomForestRegressor(n_estimators=50, random_state=random_state, n_jobs=-1)
rf_temp.fit(X_train_eng_scaled, y_train)
rf_importance = pd.Series(rf_temp.feature_importances_, index=X_train_eng_scaled.columns).sort_values(ascending=False)

# HistGBT feature importance (using permutation importance)
hgb_temp = HistGradientBoostingRegressor(random_state=random_state)
hgb_temp.fit(X_train_eng_scaled, y_train)
perm_result = permutation_importance(hgb_temp, X_train_eng_scaled, y_train,
                                    n_repeats=5, random_state=random_state, n_jobs=-1)
hgb_importance = pd.Series(
  perm_result.importances_mean, index=X_train_eng_scaled.columns
).sort_values(ascending=False)

# Display top 15 features for each method
print("=== Top 15 Features by Method ===\n")
for name, ranking in [('F-score (Ridge)', f_score_ranking),
                     ('Random Forest Importance', rf_importance),
                     ('Histogram Gradient Boosting Importance', hgb_importance)]:
  print(f"{name}:")
  for feat, score in ranking.head(15).items():
      eng_marker = " *ENG*" if feat in new_features else ""
      print(f"  {feat}: {score:.4f}{eng_marker}")
  print()

=== Top 15 Features by Method ===

F-score (Ridge):
  finishedsquarefeet12: 17487.4292
  calculatedfinishedsquarefeet: 16988.8380
  calculatedbathnbr: 11272.6818
  bathroomcnt: 10611.3768
  sqft_vs_zip_median: 10416.6290 *ENG*
  fullbathcnt: 9524.3876
  zip_median_sqft: 3853.2081 *ENG*
  bedroomcnt: 3160.6194
  yearbuilt: 2944.2281
  garagetotalsqft: 2860.0458
  garagecarcnt: 2139.8382
  heatingorsystemtypeid_7.0: 1989.0904
  heatingorsystemtypeid_2.0: 1955.6759
  buildingqualitytypeid_9.0: 1706.4158
  buildingqualitytypeid_4.0: 1628.7392

Random Forest Importance:
  calculatedfinishedsquarefeet: 0.2102
  latitude: 0.1117
  finishedsquarefeet12: 0.0986
  yearbuilt: 0.0970
  longitude: 0.0782
  lat_x_lon: 0.0738 *ENG*
  lotsizesquarefeet: 0.0692
  sqft_vs_zip_median: 0.0527 *ENG*
  regionidzip: 0.0425
  zip_median_sqft: 0.0296 *ENG*
  zip_property_count: 0.0211 *ENG*
  regionidcity: 0.0159
  regionidneighborhood: 0.0157
  garagetotalsqft: 0.0150
  bedroomcnt: 0.0134

Histogram Gradient 

In [18]:
# --- Step 2: Sweep top-k features for each model to find best subset ---
# We test k = 5, 10, 15, 20, 30, 45 using a quick 5-fold CV (no repeats) to find
# the best k, then do the full repeated CV on the winner.

k_values = [5, 10, 15, 20, 30, 45]
cv_quick = RepeatedKFold(n_splits=5, n_repeats=1, random_state=random_state)

selection_configs = {
    'Ridge Regression': {'ranking': f_score_ranking, 'model_fn': lambda: Ridge()},
    'Random Forest': {'ranking': rf_importance, 'model_fn': lambda: RandomForestRegressor(n_estimators=50, random_state=random_state, n_jobs=-1)},
    'HistGradientBoosting': {'ranking': hgb_importance, 'model_fn': lambda: HistGradientBoostingRegressor(random_state=random_state)},
}

best_k = {}  # store best k and features for each model

for model_name, config in selection_configs.items():
    print(f'{model_name}: sweeping k values...')
    best_mae = float('inf')
    for k in k_values:
        top_features = config['ranking'].head(k).index.tolist()
        X_subset = X_train_eng_scaled[top_features]
        scores = cross_val_score(
            config['model_fn'](), X_subset, y_train,
            cv=cv_quick, scoring='neg_mean_absolute_error', n_jobs=-1
        )
        mae = (-scores).mean()
        marker = '  <-- best' if mae < best_mae else ''
        print(f'  k={k:2d}: MAE = ${mae:,.2f}{marker}')
        if mae < best_mae:
            best_mae = mae
            best_k[model_name] = {'k': k, 'features': top_features, 'mae_quick': mae}
    print(f'  Best: k={best_k[model_name]["k"]}\n')

# Show selected features for each model
for model_name, info in best_k.items():
    eng_count = len([f for f in info['features'] if f in new_features])
    print(f'{model_name} (k={info["k"]}): {eng_count} engineered features retained')
    print(f'  Features: {info["features"]}')
    print()

Ridge Regression: sweeping k values...
  k= 5: MAE = $162,389.61  <-- best
  k=10: MAE = $159,196.80  <-- best
  k=15: MAE = $157,561.98  <-- best
  k=20: MAE = $150,847.77  <-- best
  k=30: MAE = $148,996.17  <-- best
  k=45: MAE = $148,827.91  <-- best
  Best: k=45

Random Forest: sweeping k values...
  k= 5: MAE = $137,909.67  <-- best
  k=10: MAE = $136,815.46  <-- best
  k=15: MAE = $136,703.50  <-- best
  k=20: MAE = $136,283.84  <-- best
  k=30: MAE = $136,288.90
  k=45: MAE = $136,239.30  <-- best
  Best: k=45

HistGradientBoosting: sweeping k values...
  k= 5: MAE = $136,138.09  <-- best
  k=10: MAE = $135,067.27  <-- best
  k=15: MAE = $134,915.74  <-- best
  k=20: MAE = $134,690.73  <-- best
  k=30: MAE = $134,677.15  <-- best
  k=45: MAE = $134,749.14
  Best: k=30

Ridge Regression (k=45): 4 engineered features retained
  Features: ['finishedsquarefeet12', 'calculatedfinishedsquarefeet', 'calculatedbathnbr', 'bathroomcnt', 'sqft_vs_zip_median', 'fullbathcnt', 'zip_median_sq

In [19]:
# Part 3: Ridge with selected features (full repeated CV)
sel_results = {}

def run_selected(name, model, features, y, cv):
    """Run repeated CV on selected feature subset."""
    start = time.time()
    X_sub = X_train_eng_scaled[features]
    scores = cross_val_score(model, X_sub, y, cv=cv, scoring='neg_mean_absolute_error', n_jobs=-1)
    mae_scores = -scores
    elapsed = time.time() - start
    sel_results[name] = {'mean_mae': mae_scores.mean(), 'std_mae': mae_scores.std(), 'time': elapsed}
    print(f"{name} (k={len(features)}):")
    print(f"  CV MAE = ${mae_scores.mean():,.2f} +/- ${mae_scores.std():,.2f}")
    print(f"  Time: {format_hms(elapsed)}\n")

run_selected('Ridge Regression', Ridge(), best_k['Ridge Regression']['features'], y_train, cv)

Ridge Regression (k=45):
  CV MAE = $148,838.46 +/- $1,227.42
  Time: 00:00:00



In [21]:
# Part 3: Random Forest with selected features (full repeated CV)
# commented out — already ran, hardcoding results to avoid ~8 min wait
# run_selected('Random Forest', RandomForestRegressor(random_state=random_state, n_jobs=-1),
#              best_k['Random Forest']['features'], y_train, cv)
sel_results['Random Forest'] = {'mean_mae': 136053.84, 'std_mae': 1178.89, 'time': 494}

In [23]:
# Part 3: HistGradientBoosting with selected features (full repeated CV)
run_selected('HistGradientBoosting', HistGradientBoostingRegressor(random_state=random_state),
             best_k['HistGradientBoosting']['features'], y_train, cv)

HistGradientBoosting (k=30):
  CV MAE = $134,725.52 +/- $1,183.24
  Time: 00:00:08



In [24]:
# Part 3: Results comparison -- feature selection vs. Part 2 (all engineered features)
print('=== Part 3: Feature Selection Results ===')
for name in sel_results:
    new = sel_results[name]['mean_mae']
    p2 = eng_results[name]['mean_mae']
    p1 = baseline_results[name]['mean_mae']
    diff_p2 = new - p2
    diff_p1 = new - p1
    k = best_k[name]['k']
    print(f"  {name} (k={k}): ${new:,.2f} (+/-${sel_results[name]['std_mae']:,.2f})")
    print(f"    vs Part 2 (65 features): {'+' if diff_p2 > 0 else ''}{diff_p2:,.2f}")
    print(f"    vs Part 1 (61 features): {'+' if diff_p1 > 0 else ''}{diff_p1:,.2f}")

sel_df = pd.DataFrame(sel_results).T
sel_df.columns = ['Mean MAE ($)', 'Std MAE ($)', 'Time (s)']
sel_df['k'] = [best_k[name]['k'] for name in sel_results]
sel_df['Mean MAE ($)'] = sel_df['Mean MAE ($)'].map('${:,.2f}'.format)
sel_df['Std MAE ($)'] = sel_df['Std MAE ($)'].map('${:,.2f}'.format)
sel_df['Time (s)'] = sel_df['Time (s)'].map(format_hms)
display(sel_df)

=== Part 3: Feature Selection Results ===
  Ridge Regression (k=45): $148,838.46 (+/-$1,227.42)
    vs Part 2 (65 features): +78.81
    vs Part 1 (61 features): -761.70
  Random Forest (k=45): $136,053.84 (+/-$1,178.89)
    vs Part 2 (65 features): +27.11
    vs Part 1 (61 features): -0.53
  HistGradientBoosting (k=30): $134,725.52 (+/-$1,183.24)
    vs Part 2 (65 features): +3.09
    vs Part 1 (61 features): -640.25


,Mean MAE ($),Std MAE ($),Time (s),k
Ridge Regression,"$148,838.46","$1,227.42",00:00:00,45
Random Forest,"$136,053.84","$1,178.89",00:08:14,45
HistGradientBoosting,"$134,725.52","$1,183.24",00:00:08,30


### Part 3: Discussion [3 pts]

Analyze the effect of feature selection on your models:

- Did performance improve for any models after reducing the number of features?

- Which features were consistently retained across models?

- Were any of your newly engineered features selected as important?


**Did performance improve after reducing features?**

Feature selection showed mixed results relative to Part 2. Ridge Regression worsened slightly (`+$79` vs Part 2), though it still improved `$762` over the Part 1 baseline meaning the engineered features carried most of the gain, not the selection step. Random Forest was essentially flat (`+$27` vs Part 2), and HistGradientBoosting also barely changed (`+$3` vs Part 2). Compared to Part 1 baselines, Ridge and HGB show clear cumulative improvement from Parts 2+3 combined (`-$762` and `-$640` respectively), while Random Forest is nearly identical to its baseline. The takeaway is that feature selection didn't add much beyond what feature engineering already provided, but it did confirm that the engineered features are worth keeping since none were pruned.

**Consistently retained features across models:**

Several features appeared in all three models' selected subsets: `calculatedfinishedsquarefeet`, `finishedsquarefeet12`, `bathroomcnt`, `yearbuilt`, `garagetotalsqft`, `bedroomcnt`, `regionidneighborhood`, and the one-hot encoded building quality and heating system indicators. The tree models additionally valued `longitude`, `latitude`, `regionidzip`, and `regionidcity`. These geographic features capture neighborhood-level price differences that F-regression (a linear test) underweights.

**Were engineered features selected?**

Yes, all four engineered features were retained by all three models. `sqft_vs_zip_median` ranked 5th for Ridge by F-score (10,417) and was retained across the board, confirming that a home's size relative to its neighborhood is a strong price signal. `lat_x_lon` ranked 6th for both Random Forest (importance = 0.074) and HGB (importance = 0.069), validating that the geographic interaction feature provides a useful shortcut for isolating neighborhoods. `zip_median_sqft` and `zip_property_count` were also retained by all three models, with `zip_median_sqft` ranking in the top 10 for every method. The fact that all four engineered features survived selection confirms that cross-row aggregation features carry genuinely new information that the models find useful. Unlike simple row-level transforms, these features encode neighborhood-level context that no single property record contains on its own.

### Part 4: Fine-Tuning Your Three Models [6 pts]

In this final phase of Milestone 2, you’ll select and refine your **three most promising models and their corresponding data pipelines** based on everything you've done so far, and pick a winner!

1. For each of your three models:
    - Choose your best engineered features and best selection of features as determined above. 
   - Perform hyperparameter tuning using `sweep_parameters`, `GridSearchCV`, `RandomizedSearchCV`, `Optuna`, etc. as you have practiced in previous homeworks. 
3. Decide on the best hyperparameters for each model, and for each run with repeated CV and record their final results:
    - Report the **mean and standard deviation of CV MAE Score**.  

In [25]:
# =============================================================================
# Part 4: Hyperparameter Tuning
# Each model uses its best feature subset from Part 3.
# =============================================================================

tuned_results = {}

# --- Ridge Regression: GridSearchCV over alpha ---
# Alpha controls regularization strength. Higher = more regularization.
ridge_features = best_k['Ridge Regression']['features']
X_ridge = X_train_eng_scaled[ridge_features]

ridge_params = {'alpha': [0.01, 0.1, 0.5, 1.0, 5.0, 10.0, 50.0, 100.0]}

ridge_grid = GridSearchCV(
    Ridge(), ridge_params, cv=cv,
    scoring='neg_mean_absolute_error', n_jobs=-1
)
start = time.time()
ridge_grid.fit(X_ridge, y_train)
elapsed = time.time() - start

ridge_best = ridge_grid.best_params_
ridge_mae = -ridge_grid.best_score_
ridge_std = ridge_grid.cv_results_['std_test_score'][ridge_grid.best_index_]

tuned_results['Ridge Regression'] = {
    'mean_mae': ridge_mae, 'std_mae': ridge_std,
    'time': elapsed, 'params': ridge_best, 'features': ridge_features
}

print(f'Ridge Regression:')
print(f'  Best params: {ridge_best}')
print(f'  CV MAE = ${ridge_mae:,.2f} +/- ${ridge_std:,.2f}')
print(f'  Time: {format_hms(elapsed)}')

Ridge Regression:
  Best params: {'alpha': 0.01}
  CV MAE = $148,838.08 +/- $1,227.39
  Time: 00:00:03


In [28]:
# --- Random Forest: Two-stage tuning ---
# Stage 1: Quick RandomizedSearchCV (50 trees, 3-fold, 8 combos)
# Stage 2: Full 5x5 CV with best params + 100 trees

rf_features = best_k['Random Forest']['features']
X_rf = X_train_eng_scaled[rf_features]
rf_params = {
  'max_depth': [10, 20, None],
  'max_features': ['sqrt', 0.3],
  'min_samples_split': [2, 5, 10],
}
cv_coarse = RepeatedKFold(n_splits=3, n_repeats=1, random_state=random_state)
start = time.time()
rf_random = RandomizedSearchCV(
  RandomForestRegressor(n_estimators=50, random_state=random_state, n_jobs=-1),
  rf_params, n_iter=8, cv=cv_coarse,
  scoring='neg_mean_absolute_error', random_state=random_state, n_jobs=1
)
rf_random.fit(X_rf, y_train)
best_params = {**rf_random.best_params_, 'n_estimators': 100, 'random_state': random_state}
print(f"Stage 1: {format_hms(time.time() - start)}")
print(f"  Best params: {best_params}")

stage2_start = time.time()
scores = cross_val_score(
  RandomForestRegressor(**best_params, n_jobs=-1),
  X_rf, y_train, cv=cv, scoring='neg_mean_absolute_error', n_jobs=-1
)
mae_scores = -scores
elapsed = time.time() - start
print(f"Stage 2: {format_hms(time.time() - stage2_start)}")

tuned_results['Random Forest'] = {
  'mean_mae': mae_scores.mean(),
  'std_mae': mae_scores.std(),
  'time': elapsed,
  'params': best_params,
  'features': rf_features
}

print(f"Random Forest:")
print(f"  Best params: {tuned_results['Random Forest']['params']}")
print(f"  CV MAE = ${tuned_results['Random Forest']['mean_mae']:,.2f} +/- ${tuned_results['Random Forest']['std_mae']:,.2f}")

Stage 1: 00:00:41
  Best params: {'min_samples_split': 10, 'max_features': 'sqrt', 'max_depth': 20, 'n_estimators': 100, 'random_state': 42}
Stage 2: 00:01:08
Random Forest:
  Best params: {'min_samples_split': 10, 'max_features': 'sqrt', 'max_depth': 20, 'n_estimators': 100, 'random_state': 42}
  CV MAE = $134,456.32 +/- $1,176.28


In [29]:
# --- HistGradientBoosting: Two-stage tuning ---
# Stage 1: Quick search with 3-fold, 12 combos
hgb_features = best_k['HistGradientBoosting']['features']
X_hgb = X_train_eng_scaled[hgb_features]

hgb_params = {
  'learning_rate': [0.01, 0.05, 0.1, 0.2],
  'max_iter': [100, 200, 500],
  'max_depth': [3, 5, 7],
  'min_samples_leaf': [5, 20, 50],
}

cv_coarse = RepeatedKFold(n_splits=3, n_repeats=1, random_state=random_state)
hgb_random = RandomizedSearchCV(
  HistGradientBoostingRegressor(random_state=random_state),
  hgb_params, n_iter=12, cv=cv_coarse,
  scoring='neg_mean_absolute_error', random_state=random_state, n_jobs=-1
)
start = time.time()
hgb_random.fit(X_hgb, y_train)
print(f'Stage 1: {format_hms(time.time() - start)}')
print(f'  Best params: {hgb_random.best_params_}')

# Stage 2: Full CV with best params
best_params = {**hgb_random.best_params_, 'random_state': random_state}
scores = cross_val_score(
  HistGradientBoostingRegressor(**best_params),
  X_hgb, y_train, cv=cv, scoring='neg_mean_absolute_error', n_jobs=-1
)
elapsed = time.time() - start
hgb_mae = (-scores).mean()
hgb_std = (-scores).std()

tuned_results['HistGradientBoosting'] = {
  'mean_mae': hgb_mae, 'std_mae': hgb_std,
  'time': elapsed, 'params': best_params, 'features': hgb_features
}

print(f'Stage 2: {format_hms(elapsed)}')
print(f'  CV MAE = ${hgb_mae:,.2f} +/- ${hgb_std:,.2f}')

Stage 1: 00:00:11
  Best params: {'min_samples_leaf': 50, 'max_iter': 200, 'max_depth': 7, 'learning_rate': 0.1}
Stage 2: 00:00:23
  CV MAE = $134,127.60 +/- $1,177.82


In [30]:
# Part 4: Tuning Results Summary
print('=== Part 4: Tuned Model Results ===')
for name in tuned_results:
    t = tuned_results[name]
    p3 = sel_results[name]['mean_mae']
    p1 = baseline_results[name]['mean_mae']
    diff_p3 = t['mean_mae'] - p3
    diff_p1 = t['mean_mae'] - p1
    print(f"  {name}: ${t['mean_mae']:,.2f} (+/-${t['std_mae']:,.2f})")
    print(f"    Params: {t['params']}")
    print(f"    vs Part 3: {'+' if diff_p3 > 0 else ''}{diff_p3:,.2f}")
    print(f"    vs Part 1 baseline: {'+' if diff_p1 > 0 else ''}{diff_p1:,.2f}")

tuned_df = pd.DataFrame({name: {'Mean MAE ($)': f"${v['mean_mae']:,.2f}",
                                 'Std MAE ($)': f"${v['std_mae']:,.2f}",
                                 'Best Params': str(v['params']),
                                 'Time': format_hms(v['time'])}
                          for name, v in tuned_results.items()}).T
display(tuned_df)

=== Part 4: Tuned Model Results ===
  Ridge Regression: $148,838.08 (+/-$1,227.39)
    Params: {'alpha': 0.01}
    vs Part 3: -0.38
    vs Part 1 baseline: -762.08
  Random Forest: $134,456.32 (+/-$1,176.28)
    Params: {'min_samples_split': 10, 'max_features': 'sqrt', 'max_depth': 20, 'n_estimators': 100, 'random_state': 42}
    vs Part 3: -1,597.52
    vs Part 1 baseline: -1,598.05
  HistGradientBoosting: $134,127.60 (+/-$1,177.82)
    Params: {'min_samples_leaf': 50, 'max_iter': 200, 'max_depth': 7, 'learning_rate': 0.1, 'random_state': 42}
    vs Part 3: -597.92
    vs Part 1 baseline: -1,238.17


,Mean MAE ($),Std MAE ($),Best Params,Time
Ridge Regression,"$148,838.08","$1,227.39",{'alpha': 0.01},00:00:03
Random Forest,"$134,456.32","$1,176.28","{'min_samples_split': 10, 'max_features': 'sqr...",00:01:50
HistGradientBoosting,"$134,127.60","$1,177.82","{'min_samples_leaf': 50, 'max_iter': 200, 'max...",00:00:23


### Part 4: Discussion [3 pts]

Reflect on your tuning process and final results:

- What was your tuning strategy for each model? Why did you choose those hyperparameters?
- Did you find that certain types of preprocessing or feature engineering worked better with specific models?


Tuning strategy for each model:
- Ridge Regression: We tested 8 different alpha values using GridSearchCV. The best was 0.01, meaning Ridge barely needed to shrink its coefficients because we already removed redundant features in Part 3. Even with the best alpha, Ridge's MAE only improved by `$0.38`, which tells us the problem isn't alpha. Ridge
is limited because it can only learn straight-line relationships, and housing prices don't follow straight lines.
- Random Forest: We used a two-stage approach. First, a quick RandomizedSearchCV (50 trees, 3-fold, 8 combos) to narrow down the best settings. Then a full 5×5 CV with the winning config. The best params (`max_depth=20`, `max_features='sqrt'`, `min_samples_split=10`) let trees grow deep but require at least 10 samples
to split a node, which prevents overfitting to noise. This improved MAE by `$1,598` over Part 3, bringing RF to `$134,456` which is a substantial gain and its best score across all parts.
- HistGradientBoosting: Same two-stage approach (12 combos coarse, then full CV). The best config (`learning_rate=0.1`, `max_iter=200`, `max_depth=7`, `min_samples_leaf=50`) lets trees grow deeper than the default to capture more complex patterns, while requiring at least 50 samples per leaf to prevent overfitting to noise. HGB improved by `$1,238` over its Part 1 baseline, reaching `$134,128`, the best score of any model in any part.

Did certain preprocessing work better with specific models?

Yes. Feature engineering (Part 2) provided modest but consistent gains for Ridge and HGB, while feature selection (Part 3) confirmed the engineered features were worth keeping. The biggest gains came from hyperparameter tuning (Part 4), especially for Random Forest, which improved by `$1,598`, the largest
single-step improvement of any model in any part. Ridge remained stuck around `$149K` regardless of features or tuning. Its linear assumption is the binding constraint. The tree models benefited most from tuning depth-related parameters (max_depth, min_samples_leaf), which control the bias-variance tradeoff that
feature engineering alone couldn't address. Notably, Random Forest and HGB are now within `$329` of each other (`$134,456` vs `$134,128`), much closer than the `$700` gap at baseline, suggesting that tuning helped RF close the gap by finding a better depth/regularization balance.

### Part 5: Final Model and Design Reassessment [6 pts]

In this part, you will finalize your best-performing model.  You’ll also consolidate and present the key code used to run your model on the preprocessed dataset.
**Requirements:**

- Decide one your final model among the three contestants. 

- Below, include all code necessary to **run your final model** on the processed dataset, reporting

    - Mean and standard deviation of CV MAE Score.
    
    - Test score on held-out test set. 




In [24]:
# =============================================================================
# Part 5: Final Model — HistGradientBoosting
# Best performer across all parts. Using tuned params from Part 4
# with the k=45 feature subset from Part 3.
# =============================================================================

# Final model configuration
final_params = {
    'learning_rate': 0.1,
    'max_iter': 200,
    'max_depth': 7,
    'min_samples_leaf': 50,
    'random_state': random_state
}
final_features = best_k['HistGradientBoosting']['features']
final_model = HistGradientBoostingRegressor(**final_params)

print(f'Final Model: HistGradientBoosting')
print(f'Parameters: {final_params}')
print(f'Features: {len(final_features)} selected features')
print()

# --- Repeated CV on training data ---
X_final_train = X_train_eng_scaled[final_features]
X_final_test = X_test_eng_scaled[final_features]

start = time.time()
cv_scores = cross_val_score(
    final_model, X_final_train, y_train,
    cv=cv, scoring='neg_mean_absolute_error', n_jobs=-1
)
cv_elapsed = time.time() - start
cv_mae = (-cv_scores).mean()
cv_std = (-cv_scores).std()

print(f'=== Cross-Validation Results (5-fold, 5 repeats) ===')
print(f'  CV MAE = ${cv_mae:,.2f} +/- ${cv_std:,.2f}')
print(f'  Time: {format_hms(cv_elapsed)}')
print()

# --- Test set evaluation ---
final_model.fit(X_final_train, y_train)
y_pred = final_model.predict(X_final_test)
test_mae = mean_absolute_error(y_test, y_pred)

print(f'=== Held-Out Test Set Results ===')
print(f'  Test MAE = ${test_mae:,.2f}')
print()

# --- Summary comparison across all parts ---
print(f'=== Performance Progression ===')
print(f'  Part 1 (baseline):          ${baseline_results["HistGradientBoosting"]["mean_mae"]:,.2f}')
print(f'  Part 2 (+ eng features):    ${eng_results["HistGradientBoosting"]["mean_mae"]:,.2f}')
print(f'  Part 3 (feature selection):  ${sel_results["HistGradientBoosting"]["mean_mae"]:,.2f}')
print(f'  Part 4 (tuned):             ${tuned_results["HistGradientBoosting"]["mean_mae"]:,.2f}')
print(f'  Part 5 (final CV):          ${cv_mae:,.2f}')
print(f'  Part 5 (test set):          ${test_mae:,.2f}')
print(f'\n  Total improvement: ${baseline_results["HistGradientBoosting"]["mean_mae"] - cv_mae:,.2f} ({(baseline_results["HistGradientBoosting"]["mean_mae"] - cv_mae) / baseline_results["HistGradientBoosting"]["mean_mae"] * 100:.2f}%)')

Final Model: HistGradientBoosting
Parameters: {'learning_rate': 0.1, 'max_iter': 200, 'max_depth': 7, 'min_samples_leaf': 50, 'random_state': 42}
Features: 45 selected features

=== Cross-Validation Results (5-fold, 5 repeats) ===
  CV MAE = $134,595.73 +/- $1,150.18
  Time: 00:00:17

=== Held-Out Test Set Results ===
  Test MAE = $135,061.43

=== Performance Progression ===
  Part 1 (baseline):          $135,365.77
  Part 2 (+ eng features):    $135,424.51
  Part 3 (feature selection):  $135,362.34
  Part 4 (tuned):             $134,595.73
  Part 5 (final CV):          $134,595.73
  Part 5 (test set):          $135,061.43

  Total improvement: $770.04 (0.57%)


### Part 5: Discussion [8 pts]

In this final step, your goal is to synthesize your entire modeling process and assess how your earlier decisions influenced the outcome. Please address the following:

1. Model Selection:
- Clearly state which model you selected as your final model and why.

- What metrics or observations led you to this decision?

- Were there trade-offs (e.g., interpretability vs. performance) that influenced your choice?

2. Revisiting an Early Decision

- Identify one specific preprocessing or feature engineering decision from Milestone 1 (e.g., how you handled missing values, how you scaled or encoded a variable, or whether you created interaction or polynomial terms).

- Explain the rationale for that decision at the time: What were you hoping it would achieve?

- Now that you've seen the full modeling pipeline and final results, reflect on whether this step helped or hindered performance. Did you keep it, modify it, or remove it?

- Justify your final decision with evidence—such as validation scores, visualizations, or model diagnostics.

3. Lessons Learned

- What insights did you gain about your dataset or your modeling process through this end-to-end workflow?

- If you had more time or data, what would you explore next?

1. Model Selection

We picked HistGradientBoosting because it had the lowest MAE at every stage, from Part 1's baseline (`$135,366`) to Part 4's tuned result (`$134,596`). On the held-out test set it got `$135,061`, which is only `$466` off from the CV estimate, meaning it generalizes well without overfitting.

HGB consistently beat Random Forest by `~$1,000-$1,100` and Ridge by `~$15,000-$16,000`. It was also much faster, 19 seconds vs ~8 minutes for Random Forest.

The downside is interpretability. With Ridge, you can say "each extra square foot adds `$X` to the price." With HGB you can't, it's a black box. But we can still check which features matter most using permutation importance, and the `~$15K` accuracy advantage over Ridge is worth the trade-off for predicting home prices.

2. Revisiting an Early Decision

In Milestone 1, we imputed missing values for `airconditioningtypeid` (68% missing) and `heatingorsystemtypeid` (36% missing) using mode imputation, then one-hot encoded them. The idea was that AC and heating type could signal home quality. A home with central AC is likely worth more than one without.

Some of these one-hot columns (`heatingorsystemtypeid_2.0`, `heatingorsystemtypeid_7.0`, `buildingqualitytypeid_9.0`) did show up in feature importance rankings and survived feature selection. But with 68% of AC values missing, mode imputation just assigned most homes the same value, making it less useful. A simple binary "has_ac" feature (yes/no) probably would have worked better than encoding the specific AC type, since presence vs. absence of AC is the real signal. We kept the current approach because feature selection in Part 3 handled it, low-value columns got pruned and informative ones were kept.

3. Lessons Learned

The biggest surprise was that engineered features (Part 2) barely helped the tree models. Features like `property_age` and `sqft_x_bathrooms` seemed useful based on correlation analysis, but the trees were already figuring out those relationships through their splits. The real gains came from hyperparameter tuning (Part 4), which improved HGB by `$770` over baseline. This shows that for tree-based models, tuning depth and regularization matters more than creating new features from existing ones.

With more time or data, we would explore:

(1) external data like school ratings, crime stats, and walkability scores. In other words, genuinely new information the model can't figure out from existing features.

(2) target encoding for high-cardinality (lots of unique values) columns like `regionidzip` and `regionidcity` instead of treating them as raw numbers

(3) stacking the three models together, since Ridge captures different patterns than the tree models and could complement them.